# NeuroScan AI: Evaluación Comparativa, Model Registry y Fase 2 Databricks
## Módulo 3: Databricks e IA Aplicada - Proyecto Final

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EduardCY/m3-proyecto-final-databricks-ia/blob/main/notebooks/04_evaluation_registry.ipynb)

Este notebook implementa la **Etapa 5**:
1. Consulta y tabla comparativa de los 5 experimentos registrados en MLflow.
2. Identificación del modelo ganador mediante Macro-F1 y Multiclass AUC.
3. Registro formal en el **Model Registry** de MLflow (`neuro_mri_classifier`) y promoción a `Staging`.
4. Exportación atómica del modelo desacoplado a `models/neuro_mri_model.keras`.
5. Especificación técnica de la **Fase 2 (Migración a Databricks Workspace, Unity Catalog y Delta Lake)**.


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import shutil
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd

from src.config import (
    FINAL_MODEL_PATH,
    MLFLOW_EXPERIMENT_NAME,
    MLFLOW_MODEL_NAME,
    MLFLOW_TRACKING_URI,
    MODEL_CHECKPOINTS_DIR,
)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
print(f"Experimento MLflow: {experiment.name} (ID: {experiment.experiment_id})")


### 1. Tabla Comparativa de Experimentos


In [ ]:
runs = client.search_runs(experiment_ids=[experiment.experiment_id], order_by=["metrics.test_macro_f1 DESC"])

records = []
for r in runs:
    records.append({
        "run_id": r.info.run_id[:8],
        "run_name": r.data.tags.get("mlflow.runName", "N/A"),
        "backbone": r.data.params.get("backbone"),
        "learning_rate": r.data.params.get("learning_rate"),
        "macro_f1": round(r.data.metrics.get("test_macro_f1", 0.0), 4),
        "weighted_f1": round(r.data.metrics.get("test_weighted_f1", 0.0), 4),
        "precision": round(r.data.metrics.get("test_macro_precision", 0.0), 4),
        "recall": round(r.data.metrics.get("test_macro_recall", 0.0), 4),
        "multiclass_auc": round(r.data.metrics.get("test_multiclass_auc", 0.0), 4),
    })

comparison_df = pd.DataFrame(records)
print("=== TABLA COMPARATIVA DE EXPERIMENTOS MLFLOW ===")
comparison_df


### 2. Registro en Model Registry y Promoción a Staging


In [ ]:
best_run = runs[0]
best_run_name = best_run.data.tags.get("mlflow.runName")
best_run_id = best_run.info.run_id
print(f"Modelo Campeón: {best_run_name} (Run ID: {best_run_id})")

model_uri = f"runs:/{best_run_id}/model"

try:
    mv = mlflow.register_model(model_uri=model_uri, name=MLFLOW_MODEL_NAME)
    print(f"✅ Modelo registrado: '{MLFLOW_MODEL_NAME}' Versión: {mv.version}")
    
    # Transición a etapa Staging
    client.transition_model_version_stage(
        name=MLFLOW_MODEL_NAME,
        version=mv.version,
        stage="Staging",
        archive_existing_versions=True,
    )
    print(f"✅ Versión {mv.version} promovida formalmente a etapa 'Staging'.")
except Exception as e:
    print(f"Aviso en registro de modelo MLflow: {e}")


### 3. Exportación Atómica a `models/neuro_mri_model.keras`


In [ ]:
# Copiar el mejor checkpoint a la ruta de producción desacoplada
best_ckpt_file = MODEL_CHECKPOINTS_DIR / f"{best_run_name}_best.keras"
if best_ckpt_file.exists():
    shutil.copy2(best_ckpt_file, FINAL_MODEL_PATH)
    print(f"✅ Artefacto de producción desacoplado exportado a: {FINAL_MODEL_PATH}")
else:
    print(f"Checkpoint {best_ckpt_file} no encontrado; verifique checkpoints locales.")


### 4. Fase 2: Hoja de Ruta de Migración a Databricks

| Componente Actual (Fase 1 Híbrida) | Migración en Databricks (Fase 2 Producción) |
|---|---|
| Manifiesto local `manifest_split.csv` | **Tabla Delta en Unity Catalog** (`catalog.neuro_mri.manifest`) con versionado y time-travel. |
| Pipeline `tf.data` | **Petastorm / Spark-TensorFlow-Distributor** sobre volúmenes montados en DBFS / ADLS Gen2. |
| SQLite `mlflow.db` | **Managed MLflow Workspace** con tracking colaborativo integrado nativamente. |
| API FastAPI local en Uvicorn | **Databricks Model Serving** con endpoints de escalado a cero y monitoreo de deriva (*data drift*). |
